## Academic portfolio context

**Student / portfolio maintainer:** Mohammed Alaa Eddine GUERBOUZ  
**Institution:** INGETIS (Paris, France)  
**Programme:** Master-level studies · Generative AI / Large Language Models  
**Academic year:** 2026–2027  
**Topic:** Document Q&A using RAG and ChromaDB  
**Material type:** Course practical notebook / learning exercise  

**Problem explored:** How can a language model answer from retrieved documents instead of relying only on parametric memory?

> Portfolio annotation added in October 2026. Original instructional content and attribution are preserved below. This header identifies the student maintaining this study portfolio; it is **not** a claim that all classroom template code was originally authored by the student. The source filename was `jour_2_qa_documents_rag_colab_fr(1).ipynb`. See the folder README for requirements and limitations.


# Jour 2 - Questions-réponses sur des documents avec le RAG et Chroma

Bienvenue dans ce deuxième jour du cours sur l'IA générative !

> **Version adaptée pour Google Colab** : ce notebook est en français, lit la clé API depuis les **secrets Colab**, utilise le modèle `gemini-3.1-flash-lite` pour la génération et `gemini-embedding-001` pour les embeddings (les modèles `gemini-2.0-flash` et `text-embedding-004` d'origine ont été arrêtés).

Les LLM ont deux grandes limites : 1) ils ne « connaissent » que les informations sur lesquelles ils ont été entraînés, et 2) leur fenêtre de contexte en entrée est limitée. Une technique permet de répondre à ces deux limites : la génération augmentée par récupération (Retrieval Augmented Generation, ou RAG). Un système RAG comporte trois étapes :

1. L'indexation
2. La récupération (retrieval)
3. La génération

L'indexation se fait en amont et permet de retrouver rapidement les informations pertinentes au moment de la requête. Quand une question arrive, on récupère les documents pertinents, on les combine avec les instructions et la question de l'utilisateur, puis le LLM génère une réponse adaptée en langage naturel à partir des informations fournies. On peut ainsi fournir au modèle des informations qu'il n'a jamais vues, comme des connaissances propres à un produit ou la météo en temps réel.

Dans ce notebook, vous allez utiliser l'API Gemini pour créer une base de données vectorielle, y rechercher les passages qui répondent à une question, puis générer une réponse finale. Vous utiliserez [Chroma](https://docs.trychroma.com/), une base de données vectorielle open source. Chroma permet de stocker des embeddings avec leurs métadonnées, de calculer les embeddings des documents et des requêtes, et d'effectuer des recherches dans vos documents.

## Installation

Commencez par installer ChromaDB et le SDK Python de l'API Gemini.

In [ ]:
# Colab inclut déjà google-genai : on le met à jour et on installe ChromaDB.
!pip install -qU "google-genai" "chromadb==1.5.9"

In [ ]:
from google import genai
from google.genai import types

from IPython.display import Markdown

genai.__version__

### Politique de nouvelle tentative (retry)

En cas d'erreur 429 (quota) ou 503 (modèle surchargé), l'appel est relancé automatiquement après 2 s, 4 s, 8 s… pendant 60 s au maximum. Il n'y a aucune attente quand l'appel réussit. Cette politique s'applique à la génération de texte ici, et au calcul des embeddings plus bas.

> **Clé gratuite** : le tier gratuit est partagé et peut être saturé pendant le cours. Si une cellule met un peu de temps à répondre, c'est normal : laissez le retry faire son travail. Si l'erreur persiste après 60 s, relancez simplement la cellule.

In [ ]:
from google.api_core import retry


is_retriable = lambda e: (isinstance(e, genai.errors.APIError) and e.code in {429, 503})

# Politique commune : 2 s, 4 s, 8 s... (15 s max entre deux essais), 60 s au total.
politique_retry = retry.Retry(predicate=is_retriable, initial=2, maximum=15, timeout=60)

# On garde la fonction d'origine pour pouvoir réexécuter cette cellule sans empiler les retry.
if not hasattr(genai.models.Models, "_generate_original"):
    genai.models.Models._generate_original = genai.models.Models.generate_content

genai.models.Models.generate_content = politique_retry(genai.models.Models._generate_original)

### Configurer votre clé API

Pour exécuter la cellule suivante, votre clé API doit être enregistrée dans un [secret Colab](https://colab.research.google.com/notebooks/secrets.ipynb) nommé `GOOGLE_API_KEY` :

1. Dans la barre latérale gauche de Colab, cliquez sur l'icône **clé** 🔑 (« Secrets »).
2. Cliquez sur **Ajouter un secret**, saisissez `GOOGLE_API_KEY` comme nom et collez votre clé comme valeur.
3. Activez l'interrupteur **Accès au notebook**.

Si vous n'avez pas encore de clé API, vous pouvez en obtenir une sur [AI Studio](https://aistudio.google.com/app/apikey). Vous trouverez des [instructions détaillées dans la documentation](https://ai.google.dev/gemini-api/docs/api-key).

Hors de Colab (exécution locale, VM…), la cellule utilise la variable d'environnement `GOOGLE_API_KEY`.

In [ ]:
import os

GOOGLE_API_KEY = None

# 1) Secrets Colab (icône clé dans la barre latérale gauche)
try:
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
except ImportError:
    pass  # On n'est pas dans Colab

# 2) Variable d'environnement (exécution locale, VM, etc.)
if not GOOGLE_API_KEY:
    GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")

if not GOOGLE_API_KEY:
    raise ValueError(
        "Clé API introuvable : ajoutez un secret Colab nommé GOOGLE_API_KEY "
        "et activez « Accès au notebook »."
    )

print("Clé API chargée ✅")

Si vous obtenez une erreur du type `SecretNotFoundError` ou `NotebookAccessError`, vérifiez que le secret s'appelle exactement `GOOGLE_API_KEY` **et** que l'interrupteur **Accès au notebook** est activé dans le panneau Secrets (icône clé). Colab peut aussi afficher une fenêtre vous demandant d'autoriser l'accès : cliquez sur **Autoriser l'accès**.

### Explorer les modèles disponibles

Dans ce guide, vous utiliserez la méthode d'API [`embedContent`](https://ai.google.dev/api/embeddings#method:-models.embedcontent) pour calculer les embeddings. Trouvez un modèle qui la prend en charge grâce au point d'accès [`models.list`](https://ai.google.dev/api/models#method:-models.list). Vous trouverez plus d'informations sur les modèles d'embedding sur [la page des modèles](https://ai.google.dev/gemini-api/docs/embeddings).

Les modèles utilisés sont définis une seule fois ci-dessous :
* `MODEL` (`gemini-3.1-flash-lite`) pour générer la réponse ;
* `EMBEDDING_MODEL` (`gemini-embedding-001`) pour calculer les embeddings. Ce modèle accepte le paramètre `task_type` utilisé dans ce notebook et renvoie un embedding par texte.

> **Pour aller plus loin** : le modèle plus récent `gemini-embedding-2` fonctionne différemment. Il n'accepte pas `task_type` (la tâche s'indique par un préfixe dans le texte, par exemple `task: question answering | query: ...`), et une liste de textes passée directement dans `contents` produit **un seul** embedding agrégé. Il faudrait donc adapter la fonction d'embedding avant de l'utiliser.

In [ ]:
MODEL = "gemini-3.1-flash-lite"            # génération de texte
EMBEDDING_MODEL = "gemini-embedding-001"    # embeddings

client = genai.Client(api_key=GOOGLE_API_KEY)

for m in client.models.list():
    if "embedContent" in m.supported_actions:
        print(m.name)

### Données

Voici un petit ensemble de documents que vous utiliserez pour créer une base d'embeddings.

In [ ]:
DOCUMENT1 = "Operating the Climate Control System  Your Googlecar has a climate control system that allows you to adjust the temperature and airflow in the car. To operate the climate control system, use the buttons and knobs located on the center console.  Temperature: The temperature knob controls the temperature inside the car. Turn the knob clockwise to increase the temperature or counterclockwise to decrease the temperature. Airflow: The airflow knob controls the amount of airflow inside the car. Turn the knob clockwise to increase the airflow or counterclockwise to decrease the airflow. Fan speed: The fan speed knob controls the speed of the fan. Turn the knob clockwise to increase the fan speed or counterclockwise to decrease the fan speed. Mode: The mode button allows you to select the desired mode. The available modes are: Auto: The car will automatically adjust the temperature and airflow to maintain a comfortable level. Cool: The car will blow cool air into the car. Heat: The car will blow warm air into the car. Defrost: The car will blow warm air onto the windshield to defrost it."
DOCUMENT2 = 'Your Googlecar has a large touchscreen display that provides access to a variety of features, including navigation, entertainment, and climate control. To use the touchscreen display, simply touch the desired icon.  For example, you can touch the "Navigation" icon to get directions to your destination or touch the "Music" icon to play your favorite songs.'
DOCUMENT3 = "Shifting Gears Your Googlecar has an automatic transmission. To shift gears, simply move the shift lever to the desired position.  Park: This position is used when you are parked. The wheels are locked and the car cannot move. Reverse: This position is used to back up. Neutral: This position is used when you are stopped at a light or in traffic. The car is not in gear and will not move unless you press the gas pedal. Drive: This position is used to drive forward. Low: This position is used for driving in snow or other slippery conditions."

documents = [DOCUMENT1, DOCUMENT2, DOCUMENT3]

## Créer la base d'embeddings avec ChromaDB

Créez une [fonction d'embedding personnalisée](https://docs.trychroma.com/guides/embeddings#custom-embedding-functions) qui calcule les embeddings avec l'API Gemini. Comme vous construisez ici un système de recherche, le `task_type` pour les embeddings des *documents* est `retrieval_document`. Plus loin, vous utiliserez `retrieval_query` pour les embeddings des *requêtes*. Consultez la [référence de l'API](https://ai.google.dev/api/embeddings#v1beta.TaskType) pour la liste complète des tâches prises en charge.

Vocabulaire : les **documents** sont les éléments stockés dans la base. Ils sont insérés en premier, puis récupérés. Les **requêtes** sont les termes de recherche : de simples mots-clés ou une description textuelle des documents recherchés.

In [ ]:
from chromadb import Documents, EmbeddingFunction, Embeddings


class GeminiEmbeddingFunction(EmbeddingFunction):
    # Indique s'il faut calculer les embeddings de documents ou de requêtes
    document_mode = True

    @politique_retry  # même politique de retry que pour la génération
    def __call__(self, input: Documents) -> Embeddings:
        if self.document_mode:
            embedding_task = "retrieval_document"
        else:
            embedding_task = "retrieval_query"

        response = client.models.embed_content(
            model=EMBEDDING_MODEL,
            contents=input,
            config=types.EmbedContentConfig(
                task_type=embedding_task,
            ),
        )
        return [e.values for e in response.embeddings]

Créez maintenant un [client de base de données Chroma](https://docs.trychroma.com/getting-started) qui utilise `GeminiEmbeddingFunction`, puis remplissez la base avec les documents définis plus haut.

In [ ]:
import chromadb

DB_NAME = "googlecardb"

embed_fn = GeminiEmbeddingFunction()
embed_fn.document_mode = True

chroma_client = chromadb.Client()
db = chroma_client.get_or_create_collection(name=DB_NAME, embedding_function=embed_fn)

db.add(documents=documents, ids=[str(i) for i in range(len(documents))])

Vérifiez que les données ont bien été insérées en consultant la base.

In [ ]:
db.count()
# Vous pouvez aussi jeter un œil aux données :
# db.peek(1)

## Récupération : trouver les documents pertinents

Pour effectuer une recherche dans la base Chroma, appelez la méthode `query`. Notez que vous passez aussi au mode `retrieval_query` pour le calcul des embeddings.

In [ ]:
# Passage en mode requête pour le calcul des embeddings.
embed_fn.document_mode = False

# Recherche dans la base Chroma avec la requête indiquée.
query = "How do you use the touchscreen to play music?"

result = db.query(query_texts=[query], n_results=1)
[all_passages] = result["documents"]

Markdown(all_passages[0])

## Génération augmentée : répondre à la question

Maintenant que vous avez trouvé un passage pertinent dans l'ensemble de documents (l'étape de *récupération*), vous pouvez construire un prompt de génération pour que l'API Gemini *génère* une réponse finale. Notez que dans cet exemple, un seul passage a été récupéré. En pratique, surtout quand le volume de données est important, il vaut mieux récupérer plusieurs résultats et laisser le modèle Gemini déterminer quels passages sont utiles pour répondre. C'est pourquoi il n'est pas grave que certains passages récupérés ne soient pas directement liés à la question : l'étape de génération doit les ignorer.

In [ ]:
query_oneline = query.replace("\n", " ")

# C'est dans ce prompt que vous pouvez donner des consignes sur le ton, ou sur les sujets
# auxquels le modèle doit se limiter ou qu'il doit éviter.
prompt = f"""You are a helpful and informative bot that answers questions using text from the reference passage included below.
Be sure to respond in a complete sentence, being comprehensive, including all relevant background information.
However, you are talking to a non-technical audience, so be sure to break down complicated concepts and
strike a friendly and converstional tone. If the passage is irrelevant to the answer, you may ignore it.

QUESTION: {query_oneline}
"""

# Ajoute les documents récupérés au prompt.
for passage in all_passages:
    passage_oneline = passage.replace("\n", " ")
    prompt += f"PASSAGE: {passage_oneline}\n"

print(prompt)

Utilisez maintenant la méthode `generate_content` pour générer une réponse à la question.

In [ ]:
answer = client.models.generate_content(
    model=MODEL,
    contents=prompt)

Markdown(answer.text)

## Pour aller plus loin

Bravo, vous avez construit une application de génération augmentée par récupération (RAG) !

Pour en savoir plus sur les embeddings dans l'API Gemini, consultez l'[introduction aux embeddings](https://ai.google.dev/gemini-api/docs/embeddings). Pour revoir les fondamentaux, étudiez le [chapitre sur les embeddings](https://developers.google.com/machine-learning/crash-course/embeddings) du Machine Learning Crash Course.